# Avant le TP7 : un plan de réassignation écrit à la main, et les sorties JSON des outils

Le client Python n'a pas d'API pour réassigner des partitions ni pour lire les répertoires de données. Le TP7 pilote donc les outils Java depuis le notebook et exploite leurs sorties. Cette démonstration montre, sur un petit topic jetable :

1. comment lire la sortie JSON de `kafka-log-dirs.sh` en Python ;
2. le cycle complet d'une réassignation : plan, `--execute`, `--verify` ;
3. le lien entre l'ordre des répliques et le leader préféré.

Sorties de référence : exécution du 07/11/2026 22:43 UTC sur le lab.

In [1]:
import json
import os

from confluent_kafka import ElectionType
from confluent_kafka.admin import AdminClient, NewTopic

BS = os.environ["BS"]
admin = AdminClient({"bootstrap.servers": BS})

admin.create_topics([NewTopic("demo-tp7", num_partitions=2, replication_factor=3)])["demo-tp7"].result()

In [2]:
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --describe --topic demo-tp7 | grep Partition:

	Topic: demo-tp7	Partition: 0	Leader: 2	Replicas: 2,3,1	Isr: 2,3,1	Elr: 	LastKnownElr: 
	Topic: demo-tp7	Partition: 1	Leader: 3	Replicas: 3,1,2	Isr: 3,1,2	Elr: 	LastKnownElr: 


## 1. Une sortie JSON d'outil

`kafka-log-dirs.sh` affiche deux lignes de texte, puis tout son résultat sur **une seule ligne JSON**. En Python, `x = !commande` range les lignes de la sortie dans une liste : on garde la dernière, puis `json.loads` la transforme en dictionnaires.

In [3]:
sortie = !docker exec kafka-tools kafka-log-dirs.sh --bootstrap-server $BS --describe --broker-list 1 --topic-list demo-tp7
print(len(sortie), "lignes")

3 lignes


In [4]:
resultat = json.loads(sortie[-1])
for broker in resultat["brokers"]:
    for repertoire in broker["logDirs"]:
        print("broker", broker["broker"], repertoire["logDir"],
              [(p["partition"], p["size"]) for p in repertoire["partitions"]])

broker 1 /var/lib/kafka/data [('demo-tp7-1', 0), ('demo-tp7-0', 0)]


## 2. Un plan écrit à la main

Un plan de réassignation est un fichier JSON : pour chaque partition, la liste des répliques voulues. Ici, on garde les mêmes brokers pour la partition 0, mais dans l'ordre inverse. Le plan est écrit dans `/tmp` du conteneur `tools`, puis exécuté.

In [5]:
repliques = admin.list_topics("demo-tp7", timeout=10).topics["demo-tp7"].partitions[0].replicas
plan = {"version": 1, "partitions": [{"topic": "demo-tp7", "partition": 0, "replicas": list(reversed(repliques))}]}
print("avant :", repliques, "- plan :", json.dumps(plan))

avant : [2, 3, 1] - plan : {"version": 1, "partitions": [{"topic": "demo-tp7", "partition": 0, "replicas": [1, 3, 2]}]}


In [6]:
# Ecriture du plan dans le conteneur tools : le texte JSON est envoye sur l'entree de "tee"
!echo '{json.dumps(plan)}' | docker exec -i kafka-tools tee /tmp/demo-plan.json

{"version": 1, "partitions": [{"topic": "demo-tp7", "partition": 0, "replicas": [1, 3, 2]}]}


In [7]:
!docker exec kafka-tools kafka-reassign-partitions.sh --bootstrap-server $BS --reassignment-json-file /tmp/demo-plan.json --execute

Current partition replica assignment

{"version":1,"partitions":[{"topic":"demo-tp7","partition":0,"replicas":[2,3,1],"log_dirs":["/var/lib/kafka/data","/var/lib/kafka/data","/var/lib/kafka/data"]}]}

Save this to use as the --reassignment-json-file option during rollback


Successfully started partition reassignment for demo-tp7-0


`--execute` affiche d'abord la répartition actuelle : c'est le plan de retour arrière, à conserver. `--verify` relit le même fichier et dit où en est chaque partition. C'est aussi lui qui retire les limitations de débit posées par `--throttle`, qu'il y en ait ou non.

In [8]:
!docker exec kafka-tools kafka-reassign-partitions.sh --bootstrap-server $BS --reassignment-json-file /tmp/demo-plan.json --verify

Status of partition reassignment:
Reassignment of partition demo-tp7-0 is completed.

Clearing broker-level throttles on brokers 1,2,3


Clearing topic-level throttles on topic demo-tp7


## 3. Ordre des répliques et leader préféré

Le **leader préféré** est le premier broker de la liste. La réassignation a changé l'ordre, mais pas encore le leader : il faut une élection.

In [9]:
def etat():
    p = admin.list_topics("demo-tp7", timeout=10).topics["demo-tp7"].partitions[0]
    return "repliques %s, leader %s, prefere %s" % (p.replicas, p.leader, p.replicas[0])


print("apres --verify :", etat())
admin.elect_leaders(ElectionType.PREFERRED, None).result()
print("apres election :", etat())

apres --verify : repliques [1, 3, 2], leader 2, prefere 1
apres election : repliques [1, 3, 2], leader 1, prefere 1


In [10]:
admin.delete_topics(["demo-tp7"])["demo-tp7"].result()
!docker exec kafka-tools rm -f /tmp/demo-plan.json
print("demo-tp7 supprime")

demo-tp7 supprime


## La méthode du TP7

Ouvrez `tps/jour2/tp7_exploitation/enonce.ipynb`.

- `executer()` et `verifier_plan()` sont fournies ; vous écrivez `generer()`, qui exploite la sortie de `--generate`, puis `attendre_fin()`. Elles servent jusqu'à la fin du TP.
- Les cellules qui redémarrent des nœuds durent plusieurs minutes : exécutez-les une seule fois, sans les interrompre.
- Gardez Grafana ouvert (http://localhost:3000) pendant le rolling restart de l'étape 6.6 : les tuiles du contrôleur actif et des partitions sous-répliquées bougent à chaque nœud.